In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

In [2]:
import fsspec
from zipfile import ZipFile
from io import BytesIO

#-----------Function to load  a specific file form the zip file-----------
def load_specific_csv_from_zip(url, filename):
    """Load one specific CSV from multi-file ZIP"""
    with fsspec.open(url, 'rb') as f:
        with ZipFile(BytesIO(f.read())) as z:
            return pd.read_csv(z.open(filename),low_memory=False)

def load_csv_chunks_from_zip(url, filename, chunksize=500_000):
    with fsspec.open(url, 'rb') as f:
        with ZipFile(f) as z:
            for chunk in pd.read_csv(
                z.open(filename),
                low_memory=False,
                chunksize=chunksize
            ):
                yield chunk

#-----------link of the zip file from the google cloud-----------
url = "gs://agntworks-data-dev/wheelsup/raw/AgntWorks.zip"

In [4]:
#------------Callinng the load function with url and ffile name---------------
booked_flights = load_specific_csv_from_zip(url, 'AgntWorks/booked_flights.csv')

In [5]:
#---------correcting dtype of the date column and converting it to (EST)
def convert_time_zone(col_name):
    booked_flights[col_name] = pd.to_datetime(booked_flights[col_name],utc=True)
    booked_flights[col_name] = booked_flights[col_name].dt.tz_convert("US/Eastern")

convert_time_zone('flightEstimatedDepartureTime') 
convert_time_zone('flightEstimatedArrivalTime') 
convert_time_zone('flightActualDepartureTime')
convert_time_zone('reservationCreateDate')

In [6]:
df = booked_flights.copy()

### Flight Operations Analysis

In [8]:
# 1. percentage of flight delayed
flew = df[df['flightActualDepartureTime'].notna()].copy() 
# Delay = actual - estimated (in minutes)
flew['delay_minutes'] = (flew['flightActualDepartureTime'] - flew['flightEstimatedDepartureTime']).dt.total_seconds()/60

# Classify
flew['status'] = pd.cut(
    flew['delay_minutes'],
    bins=[-float('inf'), -1, 15, 60, float('inf')],
    labels=['Early', 'On-time (0–15 min)', 'Delayed (15–60 min)', 'Severely delayed (60+ min)']
)

# Summary
total = len(flew)
summary = flew['status'].value_counts()
pct = (summary / total * 100).round(1)

print(f"Total flights with actual departure: {total:,}")
print(pct)

on_time_pct = pct.get('Early', 0) + pct.get('On-time (0–15 min)', 0)
delayed_pct = 100 - on_time_pct
print(f"\nOn-time rate: {on_time_pct:.1f}%")
print(f"Delayed rate: {delayed_pct:.1f}%")

# Median delay
print(f"Median delay: {flew['delay_minutes'].median():.1f} min")
print(f"Avg delay (delayed only): {flew[flew['delay_minutes']>15]['delay_minutes'].mean():.1f} min")

Total flights with actual departure: 185,891
status
Early                         41.9
On-time (0–15 min)            24.5
Delayed (15–60 min)           18.9
Severely delayed (60+ min)    14.7
Name: count, dtype: float64

On-time rate: 66.4%
Delayed rate: 33.6%
Median delay: 3.0 min
Avg delay (delayed only): 102.0 min


In [11]:
delay_flights = flew.loc[~flew['status'].isin(['On-time (0–15 min)','Early'])]
delay_flights['route'] = (flew['flightoriginAirport'] + ' → ' + flew['flightdestinationAirport'])

In [23]:
# Flight routes that are delay above 15 mins
delay_route = pd.DataFrame(delay_flights['route'].value_counts()).reset_index().rename(columns={'count':'Delay Flights'}).sort_values(by='Delay Flights',ascending=False).head(100)

In [26]:
sns.boxplot(delay_route)

AttributeError: module 'seaborn' has no attribute 'bosplot'

In [43]:
booked_flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 258370 entries, 0 to 258369
Data columns (total 43 columns):
 #   Column                            Non-Null Count   Dtype                     
---  ------                            --------------   -----                     
 0   flightId                          258370 non-null  int64                     
 1   atlasreservationid                258370 non-null  int64                     
 2   flightStatus                      258370 non-null  object                    
 3   legOrder                          258370 non-null  int64                     
 4   flightoriginAirportId             258370 non-null  int64                     
 5   flightoriginAirport               258370 non-null  object                    
 6   flightoriginAirportName           258370 non-null  object                    
 7   flightoriginAirportCity           258370 non-null  object                    
 8   flightoriginAirportState          253534 non-null  obj

In [52]:
flew['flightoriginAirportState'].unique()

array(['FL', 'OH', 'SC', 'NJ', 'TX', 'TN', 'AL', 'GA', 'PA', 'KY', 'NC',
       'RI', 'IL', 'UT', 'VA', 'NY', 'CA', 'MA', 'CO', 'KS', 'MI', 'LA',
       'WA', 'AZ', 'CT', 'AR', 'MS', nan, 'MD', 'SD', 'IA', 'NV', 'WV',
       'ON', 'MO', 'OK', 'MN', 'OR', 'WI', 'VT', 'NM', 'MT', 'DE', 'BC',
       'NH', 'IN', 'NE', 'ME', 'NL', 'ID', 'WY', 'SK', 'NS', 'ND', 'QC',
       'JAL', 'NB', 'AB', 'PE', 'MB', 'ROO', 'BCS', 'NT', 'AK', 'HI', 'G',
       'BCN', 'NSW', 'VIC', 'COA'], dtype=object)